# Visual Classifier — Fine-Tuning
This notebook fine-tunes the `dima806/ai_vs_human_generated_image_detection` model on the
`julienlucas/midjourney-dalle-sd-nanobananapro-dataset` and then saves a compact
**weight delta** to `./fine_tuned_model_delta/weight_delta.pt` (~4–15 MB) instead of
committing the full 327 MB `model.safetensors` to Git.

The delta stores only the parameter *differences* from the base model. At inference time,
`VisualClassifier(delta_path=...)` re-downloads the base model from HuggingFace (cached
locally after the first run) and applies the delta on top.

Run **`visual_classifier_eval.ipynb`** afterwards to compare the original vs. fine-tuned model.

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from visual_classifier import fine_tune_model, save_weight_delta

## Fine-Tune the Model
This will:
1. Download `julienlucas/midjourney-dalle-sd-nanobananapro-dataset`
2. Load `dima806/ai_vs_human_generated_image_detection` as the base model
3. Train for 3 epochs (MPS-accelerated on Apple Silicon)
4. Save the best checkpoint to `./fine_tuned_model`

> ⚠️ **Note:** Training takes ~35 minutes on an M4 MacBook. Make sure you have enough memory available.

In [3]:
model, processor = fine_tune_model(
    model_name="dima806/ai_vs_human_generated_image_detection",
    dataset_name="julienlucas/midjourney-dalle-sd-nanobananapro-dataset",
    output_dir="./fine_tuned_model",
    epochs=3,
    batch_size=16,
    learning_rate=2e-5
)

Loading dataset: julienlucas/midjourney-dalle-sd-nanobananapro-dataset


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Applying transformations...
Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.394595,0.377578,0.835000,0.838710,0.820268,0.858000
2,0.203887,0.296577,0.883500,0.885391,0.871249,0.900000
3,0.138751,0.289261,0.888000,0.890196,0.873077,0.908000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

***** train metrics *****
  epoch                    =          3.0
  total_flos               = 2315575710GF
  train_loss               =       0.3777
  train_runtime            =   1:05:38.58
  train_samples_per_second =        8.146
  train_steps_per_second   =        0.128


## Save Weight Delta
Instead of committing the full 327 MB `model.safetensors` to Git, we save only the
*difference* between the fine-tuned weights and the base model weights.
The resulting file is typically **4–15 MB** and is safe to push to GitHub.

> The full `./fine_tuned_model/` directory is listed in `.gitignore`;
> only `./fine_tuned_model_delta/weight_delta.pt` is tracked by Git.

In [ ]:
delta_path, size_mb = save_weight_delta(
    fine_tuned_model=model,
    base_model_name="dima806/ai_vs_human_generated_image_detection",
    output_path="./fine_tuned_model_delta/weight_delta.pt",
)

print(f"\n✅ Delta saved to '{delta_path}' ({size_mb:.2f} MB) — safe to commit to Git.")

## Training Complete
- **Full model** saved locally to `./fine_tuned_model/` (gitignored — do **not** commit).
- **Weight delta** saved to `./fine_tuned_model_delta/weight_delta.pt` (tracked by Git).
- Run `visual_classifier_eval.ipynb` to evaluate both models.